# inferchaos on a T4: kill a real vLLM replica mid-load, behind llmgateway, and measure recovery

Qwen2.5-1.5B-Instruct on vLLM as the first provider; a tokenbench mock as the fallback. Steady load through the gateway; after a baseline the vLLM process is killed (`SIGKILL`), restarted after 60 s, and the run continues until the breaker closes again. Scored with the same SLO arithmetic as the mock experiments. ~10 min install, ~8 min run (the restart includes vLLM's full cold start).

In [ ]:
# AUTO-GENERATED by kaggle_run.sh: private wheels embedded so the kernel needs no dataset or GitHub access.
import base64, os
os.makedirs('/kaggle/working/wheels', exist_ok=True)
_W = {"inferchaos-0.1.0-py3-none-any.whl": "UEsDBBQAAAAIAAAAQlDLGZMxhAAAAJ8AAAAWAAAAaW5mZXJjaGFvcy9fX2luaXRfXy5weR3KsQrCMBAG4L1P8XNzKXUVnBxcKgriHGK8tGfrpVxSwbc3uH3DR0THyacM1lGU2URHxGQYhjNEIxtr4H3li0PB6XpHmSyVstTYwnhdJHjMsiwZXp8o5mOUgLzKzLmtmxVv9nkzxm244LGZ/qdxSB+2b0dETeNcdZakzuEA6rtd11PzA1BLAwQUAAAACAAAAEJQQxRUYa0DAAD1CAAAEQAAAGluZmVyY2hhb3MvY2xpLnB5lVVNj9s2EL37VwwIFCunXsV2NkHWgAr00FvRBmhvhqHQFuUlliJVktqNsfV/zyMpWXG2ClAfbHKGb77fmDH2Wepa2MMDN45spz9vaM+dUFIL+pn8gxWCat4p74gfudTOU2MOj9Ra8yQrYR3txYPUFSnVHLkXz/yUM8Zms9qahsqy7nxnRVmSbFpjPXGtjedeGu1ms0Fmjy23Tgx3d3I9fowtF19aYWUjtB9M/XaRLOjTEM4rnBXxcY95ttKL0goXMnr11ikzPPzr9z9ns1klamqQdYYQnzakpPNb5+2O/qU/DCpUxJ853f4SD5sZ4dNCPKSU/2qPXQjxU7jZDGU7Fmx0yeYR4ro9QG3Oq6rEOUKtyyrhfMEOTcUWZMU/nbSiKv62nUgoCwxeR1SCZAwtxOMHodoinNHBvn94idQnmtiHYaMp3oecsdtb03mYQxmCCVhMhXsb0JMYizEAyJ9aUdTKcD8a+JAvL8EhH6TnCE0kJw4GM2Rqcl7w6kSAVZMO0KZb72t/27gJPx+Xyx+CIRB+ArvM7++nwAM1JqDrSa9RP1WUSZRFWZ6EPf1vIHhrvFdiukLr5Q9K1MrHqRQ/Tk6KVqdvR2Uo1WKIZfEolVok28kIjzMfJzeYchmInwembVebHck6sOiJpEtcE8qJKOkpA64WgaZZ6mbBA33LdEHsmI/SDcJwaRy9pdWYdhpl2Hg5R0FtLGneCCwE4nlIJ3etkkhuweaJ2ZHd4AxAl4WT6WK9oANv+UH6U7H60PtuXHGHYZdeheN6fmUgd4jTZ6PQ29PoInyw7uBlXHFhczwtQtZYBSAYMgs/UTLkd53tgvZW8Edhy0bq0vGmVcIVq+Uof8beNs+o0ur9KDwYoyDVZXx7FdOrjxeaa19+Ew+9oVUOa71m31nnR9W7+ZW9tJfSDg3tjtUvCrrMDktND/IrIHoc+saG0WIbdGy4xNRZmrMgj6dzjqnIgqHrEPop2AbNDjZR9hxrM4uhLYIjfOWXWcZ50OQDOb8zWMLKldUrdWslelmzl6Da3LszXfbgiy23N8PtZre5q84krDW2V6UzFO+guFvfDwiUtlSywXKveuW+q5BtUqczuqodqIoXb0CBzft8WZ9/IvYf7a1Zi0GADauThXANjdSxz3ARwV/ItEKLqvfSD0+Sldzf7EJmqUBDZulWIoUzFkUtlSgC4Z0HjexYxVpqrtR3dOhJY9qeM/hfQp2v/s2z/jf0Bur0LtUb1y3zfI9J2c1nXwFQSwMEFAAAAAgAAABCUBw2J3VnDgAA3SoAABgAAABpbmZlcmNoYW9zL2V4cGVyaW1lbnQucHmtWltz28YVfuev2KIPAWIIomQrbeggresq7Uw9jsdW+kKzGAhYiohAAAWWkjgK/3u/c3YXN0KOJik9YwF7OXvut4XjOD/Eu1wJ+VDJOtvKQjUivomzolFiWya3oqrLuyyVdSOu5SYrUpHn25tYyft4L9y3P75/e/nh6lOwTUUjE5WVRSNenlx4wWz2od0Y11Ko8lYW17JINhpsI+s7mssK0eyucUgim0Y2wm1KURZSJHGBA8Vtlucy9QJxtZHCHlvvimaGjbFQm1rGaSDeiEbhYS/KShYneVlWIi/jVNQyKeu0YYif3v0o5B0IFKAUM//dyUa9BpA1MyAjXH4GDTIV67KexeIe1Jb3r3FIe/RX4AJOvAWARmGIdjXxtgKOhEu5u9mUO6WxbXbbbVzvRQMUQBhB0Sfhafbp46UAvIW43qU3UuFPXcjUB2rxLb+IGuB9EeeyVo3PhIBfe6Ego8BxnNlsXZdbEUXrndrVMopEtq3KWom4KEqgRpKYzexYsy+SrLSvPzdlYZ9raZ8aiEWq9q0VSjuybx8127Piph0AVhqhNFZxkscsSzPZDvlinck8bdHaKFU9GEKyYi3rZBOXTdDkpd3qzgR+kJzPD5ckPv2oGROtsxps4xHNySgB4bttN1gXEbEygsJlstGjlpsR4e3PvNns40/vLj+JUCwfnSLeSmchnCq+kY4vnLwsbqIGIy/neGs2wItfL/BG4PF49ip4dfBFt1dlxMvB7m+Gu8/m3faXh9Us+tvHyzf/uvwIHGoZJOW2ynLp1l911hYZzYtY8z4/WsMMHXf5H2f1wnM+H4T7OfW+8mbR2x9/en/1RWju5/sXXqSgK/nnR/cvCwvuFxLHL0oWcaG8MewXDPzT1ZurS4B+nIPUJC8bmYLUM7yQ+eHxHI+bOF9H/H6YzWapXItoXUsZkVxdT5x8D5GrBYvjPlMbo36B/oMFMSxLT9OvCa5hja7rnJ3/KZjj3xmOmXteu6CWiqymCaADBIMk4XrLs5U9/D7OVLSRca427q7OFzDg2mfFhcUuxBr+QoGm8znjdl2WuT49habnGfxHqG1vW8K+yiJLgOMLu1+TsQGTjxd914LoyFH1vnuhX7bW1kDou2vnERgeTjW2TotlOA/gWUn8uwZ6ngInQng+BNXjxlW9k+2cfEhkZYwu+OfV1YfLui7r4V5GvsmlrFww+cKb9YD9EOeNBDf/2trzjP8XraPXsOCc3o9DBwmz5+VfCxkjEBgfT45TKXKhkAB8OyEhytr4fjg08uOguoZrZtdHpxQL0h8SGL8mcRUnmdrb0bNveFiptYq2TSfeV3Mez1Q+GNZACD8MplmilqwdHc7BB1LlFZayC3OhUuTLo3WcqLLeh7RHcwuSG8JQ9a9t431/xUkITGpvtG4tSIUbt5H5mlUyzxoGuVqMlX65dqrH7OBQ3BIZhdQ6Lm4kbw0KbzVrQUZNFd8XPOEzfDYDBv++7Csoe9+QmKmhEFVL2rAK6qbKM+U6C7J5sjCPtJfmOJjb1UJCXQY23wJPkC3A1SKgBPJBJjsVX+eIdc7JljxmlykEpEU6UaCJkxOC4zBLXXr0eNCK3kwwAnbM88emgQ2kFCfbpr/e6IkGCOUYTWt1ASNbd0SjrC6aKSBnrCsuqCQQKRlub/Lvl/9+/9O7dzwl63piyhse07Ge1Mgh+12cnrZ+cPFIrDg47SaKvRDewOGNAIFO+Bh6PEBTUwFnJdjAnE5V+L3TvqF6kKIVrbhZUYeOhMeNthVeH2pZfQlo1QJl7gZ3cY40zfWOvGUVVGWew70iAxuCaVU4ULLeZgXiXU/3nn/IkY/WQImtrnXIF95ghXGxPZFe6YWXDxUlKlPwyMu5PQ5Zbzg2Uh/QVR1HOegpkn3nviY4qX18VTYcS0bCP5zGKdhyikRpnd1AzSkdDB+dMXgkJuOhQxeKLnooMwm/7lPGRmNJf3J+zOr2QBMOnnum1UOsAxATtS7bgmfRuccoK2A0kQHcxq9FF+Ngt3m54JSUU/ReaAnmPBkdBZ0/zzGjM6pouOdsjqkjpbBrkR42arTY1h8RRBjp0qNpg15vWhcvUQ+Ls4tuNkF6g+miP38xgcg2fojYH7dnnL/yxhL19V+Qbp506cKPHQDsrQZM9E2J0y3pjJQqgi5XDeKqshXBP/TQW1Zev5WLfh8ilqxvcOhggzsgscUnfCwWI1CsLWEBlsWNjGA94Q6oIufKQ4fCkuNpJ+iLHXmTFhQbGgKG3MKhHIYsRXmTFSHFcbdbz87TG+hH2Hv2B8oQ9l8062KFsyqUQlDnqK0Jzocno0xqlOZyZAwKGhr2tFV8Lc4H6tufPFarcDwwPG9CScOJsSmFDI+HumWg7xql3C7ReWR4bpzCH8UJfm2DIM/WMtknyMZPfs+vlzaRu4kMeB2/Oiswmrm7y1DpF89R4gTkoCTFyDjhmsyXdAqEWXNE8IkHXPtqNLaD6lr9h15tEAXCQdlE0EP6zxd5eRPl8o6UWlI14PQqKtRkYVfpB1f85IIRKFFCjVKAbN1HfS9R74RUbPR3B9pHd0OwjN+ewWAzZWdWwqOsxe60VSBjB9PZ+HTqQEeoDwO0f59mTGoJoJrAwUk+hyTxCwckMxSBeaF+J7dCNeHAQ8/tut4gdR3afsVgnMMdVRCLkaZowm/uKSvV8W+gvp1EuB0GmFxacHOFcszlqsdP7p+ZFXTWaIE1TJU/vYbcGHULHM6zkFa0QeLQIY68cKBujE0PVTWfKMC7WdZKS2+lnepybiofrVP7ImExUWNwnOvpWe5D6NzpDQ28zTNCoo0AN/dd/kNiybNtpppQ73jHLy65ZGRWhWmGhhS0Pe5mJMfZn4qbW8tc06MLrjBmGHi0viMCOgRdQySIbYLgHYNnAZXpnpnPsYt6Uxy9YErgUBPfSMrzlo9OXebcudo1utgCDYg1ihthkIdzWNGWNlRbKXYj1N1SkN4WU9wqwEB2U5S1jGRJ68k/HCZxVF8S7WDhVEpuf+QgY3IZItGpr3N6d3aKmKtOqQGWSxaIzXeJL3CO0DaO/87DiQ6tLQ8O0ygwGoiK5MmI7W4dGCDcugGczuX3Kk33qC0ET+Qh6Br9OOWHJ08sb32h+z7U0ZtoAfnD0UlA0w0gUs2J6qlPqz88XzswI2O1r6QrvSCKyOaiaBJQLXO2TdZXonya0mxtD/kD0unzb5/GSjsuCqmySF3tK1SIY+D2yjINbUIa0JtrafC8aZkaL2fBPTqkBQzM4QSItFfDcMpbPBM3HI0p2QE/HLxjcy0enqXZum+oo1lZBVkTNdQBnaa+KO+fay3UlcHq70NC5AsSJidkabdeyOQSNOeSrwGgp7hnKX2BABbMocnk2ZfG2a+mNRGq8zRC2ogtJrobiaSROpJnPh91IhifY/842Akr3CBL+poJ9E1uEGkrYN+sM5YWCvcnODG1rfajnsCzRfWkp9pCeoNO78394XQrVZ0lTa/Te+4FCtX3JAhuYSDRJ5Ok2sNeHQRr5OIxSurtE7pDvy5gD9V9yj/RFYWtKLBEn8mKT6FCXwMsuUWovNWEv3xOx9n+KuonH3Gx346+6IkqV1/ITfN4e53Gi1YPKC/TQd97Ilm9/hK4kUo8ASJXNt99DXBHuW+1ielGbJAX9ajrskFs12uteFzHTtKNB/UYJkXVswadQsKpcdw3V1I8SA6NiSKB8wNG8NfOU36qJVw/qRNdCG+b8hEnpBQWuppobS47EZ4c29JyJnqEVdA2vCxGOq62CNFVwDlHyS5vzQfwqY80DZs7TBruk7ubKrsdozZwYTat5Dz26yn0em3AnlBNKj8hmKUDyTLgSRaPtKbTBAbg+H1AWlYr/wh4j12/RxjQuN/Ad9ugM6x/PdaVL5P9ewQ0JYiufJqwLjvpHLPw102NowCHgNfkAX4uEaasC/+mxzjj4J/n+y9Gvt/U2s2m3OUp4ldGHCTvQ58mDM/sKWJbCnMByN8iZA1SNc6cfJvz+L2A4Fs0fcOkHkf8gQS9/o1SC1rXvv+fA45LW5PyNdQauZV74+WRugoZqEGvJGNHO775bynvmpVtM4sK70bJih5711TlDjVQPbjN654Q91akeY+dQ1zzpWWRojKMryl9vOPwbO7iu/CsmTAK0vY0UidzTehqYI8Hb8kAV+ZG7m6i8O3prnHxXaCh6yhJV24F1Mp1GyyB0RC2/P1NJyDOwJddzF+R/Zkj6Bq2WZrozxP6th8xlTx/TwI5eHotAV7ytdjWPeOzar6YNB/kWAMfODHxHYobjRyeXIM2ltqnhlMvOk7fLp7Jb5/OSYW+OV7aHH412D+e+77VC30Z2HPbWRGxuoIao3T6xmFZT9LFTK66iyYT+unOajlnxvFFxPJsQO+zt52vVmMbfxzwwESJRd+2HA2NSn2GNjplZVdE19DClNYZAx1CNh9N0XwuC9cQTX057h5yNfSkvIn1oBYV1Mpy39RQK1PveePjyHq5ySLTL4LugwqnQRnMIwizzac0FVa8vUSLEsXBFsvS4Xo96o3PGn2I5CzGnyYduaIxBPoKLGodGABs4we2225sRXnH3OusWHs5vos3PpBNhDLGMXqAoZdwF4c/i0IayB9UYaT/ZdWE0+QvpY7ZayMsV869r6smIGi9RISFuz0Ouc0OooQQaPYlko+Rq+B0dqVp40bEiDjjy7THQCShcp2fKa/tPJNN54kD3ajfASDcjZL0AthITMZPRv3QxQzoXv2xbeb5dZzcWlNpvT43j7pZ9vrtxbRWR2VFdrTPzo13Dc+2X37ZvQBlH8mCWU66GUhEIKpSv70sU36hhwNrG3/moVevJtSgTq0OUPRv5w/mKxey0yxBCNyUaXfnXUbUlINDabhxzp/FUAqASHv0tQutDNLdtmrMeorFKdAJz3xhImdIt8Gz/wFQSwMEFAAAAAgAAABCUHUxdzboBQAAVQ4AABQAAABpbmZlcmNoYW9zL3JlcG9ydC5weZVX247bNhB991dMCRQrp17F3qRA6kBF89A+FgG6L4UrCLRF2crqVpLKepMs0H/oH/ZLeoakLF82SStgV+JwyLmcuVkI8bZqrSHZ5GTlulJUtJokGWWpLaiQfWVJ7Tuly1o11sRCiMmk0G1NWVb0ttcqy6isu1ZbXNK0VtqybcxkEmjvTNt4/k7aXVWuB+a3WE4m2W+3b25/zn6nhD6KTdUalYslzWckdrIqsrZTDdYLrMPnzeNkMslVQfe6tCrTykBDE4X3kvJyY1fG6pn7SmfU9jbLS70kEOmTEzul6x/d9nJCeAIHVODNKCynx3txfYf/USc1OyG51b2awS2lsVl755aenZ3XyBqbmsqGglYxNK1NNPXi+BmE0HMqxEc+8Rizp8Q09nZZtbcRU+K8rzvYN8N9OYQn8AXMZ1wSWDT1cj10CQSCSWe11Hd5e98MbvFMR0LFpq1hTMkC6vxUqrvLn8g6Do7hloMv/aZWQL8BbO4AoHHvAZ7PaHIBkMMCS++bnZI5zFgJZ6AA7Fr92StjDX8rrVvtvl7e/ODe6z7fIlI3iLi+RuiA1Cl5R2vWTEurmCIrpRHiRak9x1qDRWniiFI5ycIqzXRTtfe0VsBQUeBhciGrai03d8ars2nfK/0gUqdvVTbKOIU/kaDv8Id3/K4tm4hNmXoSH/zktq+vr/HxjCoVGNL/FTYFQzw4Jz2QgyEJx5JeXQXdM0/OpL1K6ZqK1ZWxUmOxjBfFIxlBZcG3XbCLlEpDSGX6tW2UKw3FSrjDZ1uqMorEP3/9LQ66BGcnxEZ7T0Arubpi+67SR/qJeOXAGFSLRt0InphPl/G8eISCvhaxR1YBRYGAAVE0ED8KBSoH6weEMnNh6bj1dTscsrHs4JQ8ukR35eFC4Ebu4hCk6fRAC8F6ROF4zKqyBqy5owe4XAxnQwxfpc8W87nzwLdcgu1Oke6bK0OeUcwOOn7t8fdzQmScEBkrwE7B1XvA4z06C+FzUJPTIPNpkA1pcGTFmA5MhEvTIcpPyoL4owmucp50PCCF6nBaWL5UtQ/1moHymRAaSC0t34KWMnHkcR33RkXizXYbVLo4EHcP/EXSUFdZf5wB5bitUNRP62ZRbuGsvdsFe2z6tVefs9gdgyNQlsFnyg8qiRY3M3oRvwx57jlCoQ7XrPidcliOHJQktPCh6HRglrGryP3MqcjZ8KHs3K6nGAcN3Ob70FgttGsIzhAXrunrzxUQoOi02jtR+5BxnC46d7lS0H4lrC3seeqMd8h9bDbSopZGK2YG33gZ7geoq+NLnhHCfH7BY5LvZ6jnVasTbizLrZYPLla7nUzmMTYruVZVIoCxptvbX24pqs00IM3P2reQL9jCykOV9g5Lrm57V91sz9vfJITm8t/MgiS2ap4GpLHmPEkWL09M8G2HO6EChTMvmOBKxJHmLAuV4MFtR8IZVxsxfX3YgSoVQts81FW75YvKxu6A8S6BM48vuoELcMjel80+OtmIOXajlRltMcFHrkbAq6WCI9gwE2iukfrS/MUDRza3WjZbNZp6aMgULea4AHphLDiGzW4tw2ZOJYTeZDF9ch3x+EGvTrfvy5wrE6cN2hPaBxfHdEQOjLhzeVIt2X5jVXdpP1jZ5GEe5X2OCUj0oXrOeb9TWiWiaw2PKUeWr6ve2X2fLI6idZg5ojn5KRdDGC7lIRdfN3NXhKfiFKnjYBgd+PwwnEyPbX26RZ+bH8v9e9NJNOUD+4yPosuJ9ImoHfJu/moU9lRXfkoOAxZJVJvQ688EdL3uKucqk2AwCj67yAZbWoT8MCUvQxekr3TOGY1z4OebIGZ416Udi//EiOK6mhsU6Hyg8IPEGckcDRG67TEvnLHMFlOeQk5M2+oyj7x34xc+wyu15WGjajeobhg8NIpKwdFVtPjRwZ3llWO8eYJTl9vdGeuh36yuF6lz5T7EkoFuTT4ohLYVWz6eVfIBfTeCEKYZ+V7hffyrwfUNE3cN1568K5PFzRzc3BRdVEfgn07+BVBLAwQUAAAACAAAAEJQVI4AH+EEAAA5DgAAEQAAAGluZmVyY2hhb3Mvc2xvLnB5tVdLb+M2EL77VwzUi9TIit320KhxsMBi91TsFkhugqFQ1igRQksuSW02TfPfO0NSsmTntYfqkIjU8JuZb150EASXf34FoWpzu0VTbyD8+PXLx09/XV0m2xI0bkzdNhqWMfwSw29RCjdtW54WogT8ho3RMRSdakAJg9B+QwX3dVO29zqeFV15gwY2dLzbYhnDtpOmnrvvICQqPq1ww8cewNRbTIIgmM0q1W4hz6vOdArzHOrtrlUGRNO0RlhzvEwpjNhIoTXqXmjYms1mH4ZFSOL/YLO6Uh1GM7sFn9j8dAb0mBQq2QpjF+xfCkXbyjchiDkPIBS56lFgBYvk7AyG5yeolLA8QltZfM8dOf93VyssHYipTK7HIL/DGERYcdQGau1Q6gpqA7rbbBBLLIkh2iPYq6vPVyx0T0GtG6A/RAeDfNipdke8P9hViRW4IIUaZRXB/MIpd07xo9BwdJfJAubAQolzdTYAsCH2eHzgwL/wpW0whvbOsWnh+eUInTfD9i6y9jsQtp6ibSHG2+crb4VdUoDYhJyyMe8pDh21Kcham8wGeR2DJrP7+MSATenf9z57e51xPkdXkCFUrQIEotHHjEi3aHAOmBi2iODW9hh9YqPd6SM3GX42WlNVhMs9vNfpMTBhYiM4BYlN6L5F3l8uuJwLLteoatQvuCzblDM09shDZJgN3A1LS4E9WdYbs07HnnjgQ08y565ZUMyXxJITyxbrhMl1i/mSVlas7WiXEzojq8wCTrx+x/RtLREsjQR1Akucn+31FRUdey68DDgfHKNVNJwhdYnY7Sgo4WNgghRIezCGoK2i4r2eRtoISdMpU5a4goiYAdobpyFKjfbtaa/MwMlq7I7njWwAiiv9Yx4yMmNN6WKW6aSe5b140MSXkB13T3OLfRKQ8XVzA5SVvEm9xofiXcQsp8wsX6Nm+T9x4xOFFA45y0fzfhi8nrKHnYimwudRB2VOUCki2I+YTlPzKx74A5nh6zTkpkW5+Z0OSvvNi0d2yryd5YtkYbd41q0Oq3XfDEbVOnadT51CyNXrZCP4eUyi58WNwbziKfBWHatOYv/V1upBJZOV5POz9Uwe+9PZYyO21JZl29xwfuhbGpu5H+NPa4IiU6gskfuyue3HOfHIQwauWez6D9hIFMqJ2RQlNOBEKGnAaChQtvfvpPnHmoklasyB7Sv2EyfIN0yBtzNtVAy9wOOTU0GvP9Z/ONyKw+3IG/b5kQWRV7zan1QWOJoD7nxR/Lqoj4STnapikEKRLllMSpGppa2jSuxTt38ctEXQxwj6HQgkRiZy6lArY1EixRFux3Nv4cWKxThHSIw/DJrHX6ZE7mOXDSps1Nx7uldMjcnGPxd9X7dp2G+4+0awQ3GXW0Vpb9fTkUKL0zfDI+1T+lFOvB88f4cb2cga9mkrvh9rmwjFvc3REToH2/N8/hqZL1gy4opNMc+eK1Een31x3PkStmT6htbf53O+z7/V0SpBPwlyCsH4ctLR9apusBz3tV8XvrEd39Woy1ySzqbUYH8U2OlAmecHhVUBXWNqCdcj7GsW8F2c78mU1DD8qhlal6gMqhfvgXz/o7weeTG5B9rDL18DeaznbriMEFyX63UdQEzGzUE72qNhso+tzV22c76XuFhNSJ7AeCPdiUOzxh78B1BLAwQUAAAACAAAAEJQzhSwzjIMAACiGwAAIwAAAGluZmVyY2hhb3MtMC4xLjAuZGlzdC1pbmZvL01FVEFEQVRBxVndcty2Fb7nU6DKpPrJkquVZNle154kspy6kW2NrbQXnkyFJcElIpJgCFCrzWg6ve11X6FPlifpdw5ALqUkVZKbakYSFwAPzvnOd36AfaOczKST8V9Va7Wp5+IgeRS9lZWaC13nqk0LaWw0zO4ns2Q/+tBVlWzXc3FCs0LVS10r1ep6KXLTirOzN/5lVacs5zuVOvHV+TfCFa1xrsTCiWhVU+pUiitdllbIOhOulXmuU2EbfaXsBKtVLSolbdcq8eHsnVh0bc0rW5Waa9Wuoy86V5g2VpXU5Vy8NZWsxV9kLlvxp7rDh+/o2Ray+EFmny9pVZKa6kV0plNVWxWf3jStst62N68vhvFXuoTmZ69PTt9+OI3eq+87jXXx+RrbYemL54fJbH8z/lJbNxeFc83Ni+f7ycHj+1OVdE1pYPmCXn0SnbfmWmeYPr2B1XORqev7r5RltZROreRafC6W2n1G4u18OsVz0S3IjikbzPZOR8ufCUVSxfPnYhuCtx9Q5qHlzdop6148f3Bh2+U5mX/80EJnrlS9ADuKX2XaaPlDkrtrnZq2Zic8/cni+6gTCvY3oMPrt6OXyqatbhxYE5+Y2qnaxRfrBoRxWDtFaFxlZlVH0SfjGIr+38GSRAQAi58dCZOL2SNsJz4Ou8d4amXcmNblptTm253/7ZdffHEXW33yifhbIR300ha66DIDBK9/n3VQ0xkhhVXtNeFmnUyveN1DBrMWa6EdudUhi0Vwd6nlQpfaYdyKBpwAcl2dqVbkSBAQNxG1cYRNpipjvZgPTroOr78x6VXceCK1wnbaKZGZWokdUKgrHYxVpVntPiOPiIsjREWNRyARrIN6shTX5Pfe7kpncWlkRvpg17zUy8KxKSu8A4FCZpnKRqj6hQXgj6Jb8cZkXanErTjv2sZYPEW3cRzzL6YvNyxMbGkusfASgF2KHSfbpXLiM3Fx8epCLAxQ2J2IpTHZdAF1FJBx4BYhG7fILuQBDX8QvLBkpWsQneYzEpOa2naVyiaiAhA69tNClqoFLDuNXCoxFU6nV8rtTgY3YaRile8oqm4a7FVBAdY3hG5rSetNPqjgDaYFZoS08McCvkmR0pWd9xwDfI7cQPtM2A20OUjUkhqXp6OddkZpFABLEqEkLLIO/9bCNKqGq0wj2F9kQpsRJYkBREKGTEAg5r7vsMkEpAIaG6wg70q1MbYnPGXV+Biwvqbu/gQIkARxRSCc4zUvjCzBayBdB+PZe95Lgrw0bCJ4k0nwwK6PLLAScdzIVlsDZspFGcAH59XCmCs7vZLLZan+PlJCN+t6QTpcFKy0Ehv/wDRX/CyvDSGIEJA+zDhimn3vApX59z68/urr12dnIXDZKf2UBmtYaGrKTPAUNI0uEK5UkawPsdx0bcB4o5PlsFNkr+F/OxLsduLJviAcllLX1olSXytPoV49it5CB5h4748bRjyUDzcrQwL8xoLyUXR5ebmQtoga3QjaVyKmtx4uexv4t6LNMxsWx2RMSDhTtiD2AXqMB2zm2XGwj08emmN6HCLueJ+UYh3feyHzX8BhFA87x8TpKfIBiD4HYVOKM3Ycp48f//Uf4LsvKkvx9vTpp7vw1cqMZOra068A/GLnstm/RPg1s0tE4eyY0kfatS0xKsQOqqVMi4k4YqG0yTMRtJmLXLfWxZwLOB4IkVlyLOyG/uAfa9Y8fYTEo9oW9ZZRCvkLBdCGwPSMm+37iESKq3Stq65i8yRPQEHwEIX9mQ8oeIIW2lQSmcEzsm0UFpzw5qi1ydGN3/GQpJBA7PssJMK5OAyzx/0sbbabiFPYTt6ekx/hkODViV/o3QoLPYw8NhQ+sfXl3ZS8RRWjVaV0RPmga0jb6ATo06owyASzIxbU1chIRjw6OPyUamxtfajxnpRnYHBOkrDQbosVElSRUCXyK243Drz1sNPD0cFT+nevWGCkgbc22QsDoWLAwzw/uBPJFwMyRyHHMErZCjRFE6WGJaRBWS7QHljWIvB9VA9/zS8MGWLoVjwBJLdi+P20Bwy2b9veHEzc4E9t+JUf//nv0Sv8CTKHckQyZ6MFDPPPSz1+TGI9VdAsz8BurqOfiyOy8HEyE4wtiTk6wp/D5IBGsB1lWd7q4Leq/4jCKLxweHxHKDdkNHJ43FtwcHhw9PuAOUz2WWz0h49MHPvtzjinTf1g0tTLXcpVQ0epBA53C8ooVq6jaJaIvT0uTMSIodJoT9pM1kvVms5SjfYlI0Mt54qNJYVsK4SxTfb2xNdAjIo5hRTJyboUfNvb+wHvY5pqbXytraaSGWhN8RoCYm+vplXe7jl1AmAeaF6j5cVRgZtLLOMKULtyzT17n8/AVYfWKmOBuaKuj3pcykrch+RqJQq0ZxQRFalpwe06Ax36os4K+SJwL2C8O3WdoHr3Tbe3kbtqcq3v/7wdv8hGYC/vByvWgaJzIuAQ8yu0pXeikzth3uGuYp693NwP6QXZW6USUHPuD3pUcBOn2JXUngB5B4g43f9sIQgbUzIgYwlLjxR1gSXah/qOZG7gbciKQUcar41YttQVW4YqdYl4u17KNgMoW2xhpmBKhl6U2uKiQ1WovMWyHmhmt3ouhuMKNtR5ODtBVbdS0Afxw6UB6CfRQU/poM02d1ALsgWieg7S+tQQ5BK9DKcWBo1QID6/9vV2yDuNrFV5z8gyj8kbNvD1TqFDDVBkO6FD2ysaUHbMGd+2eRHP/O68ktRkUAkkblIDPdC+FzRrUIsICOO4XfjHLHnMVe+VAn0mHkUqprWlOsMyYThivqPaZx1LDOk9oNurTafHDDrlufc5KGMLs+IDgmTAuEFP1ykwoTNUEh0S3u97eSss1BWKkyaKI/5CLKMmAFU24JCJt6SzZ9g9x7Gg7eOmomhO+QSi6NTEsUqTUN+h+4HldMoKJywhW8X4cMED0zKVln7MawRc3hFX7niz1CiEmc8itbpxAXjuy1QWUsnY2WmJs2HW5xTyzqajMb340JsPWZSwGPpyPhgXxEL0cEirGV9hkHvtELb90Z1elGXrj00193lJdMQ4E6qlrrQjSc5nHdrbFxfOBwcTqiqt+s4nT59oKQ/3yYhSEHGcgkRmkEVsGlJQTR4Tpcp9tpj5tnQBIQmKiMIY9X/oSMh7iECiE8es9IxFx7PyJUSGQuDIKZpuCumk5YhUfYEJ4+GUr1Tjx9Elj7Lzpi/0XGRXojtGtkLfD25nhm8dKo1sGmzkmxNGZ7EejIwhMiYl/S3FYh1QovuBR33auNNBUbhs3MONH9I3+YeutlTlSe1bDFgwE32TgQ9H1JPmHp4R9xZqKQFEf5A63Edh5xa0NHCpb6nt7hzoUQPD++Xh+mO0YttygACQMvPH7byVvlj2TSmvS8QXLhwwtYWi4gDbPQpyadlIJprRDtIcGE89fIeTIkUUITFBj6uRoILfGgOmDQE7vrHIlNXLOomOCc539aZIL7ol45ZTTfClWt949/A918bLc2BKcrlnyGVKTOeEWSAgOHfRqb7nvT9KoaYJuqWqGrcd0uuOP/qYOhRI39vivFRovnZgcnMV7DejKtVh3Zprpcr6d3vXbUI7HJ+pem4c62+rkBN9W+3ZV5txC1CWCUddsARm18AMhQI1OFb1Ju8AizjYM8qVVjWSWgcSIocguqLDJvsiZBecdOmQHyoetPFH6jO5RsTwmTo6eff25PT84kNSZYJ+vJnKIFWPTvGNv/OE81uKn3rclKQmU5Ft09FZewpBmo6A5EVJXIz4smEqhp+T829iUwNhnpgQZjVfo/B9RykbZ5poc5viX6I7z2FM7HzNVyw4lZ6gEC52fe4I4N+5x+BbwJZuGfvWOOofen2GrqLviukdf7c9nPKhIqAc3UXwNxkirlBT62uR8N8//hFNUNeiEvDnKfqNKcXjNdx15/YiVmIr+Zip62+3Iv/1gN+JjLx/CZO3phpfLl2GNqEfobJUU5AWfERtzKTfxlKSHey3fB8E5zed45C+7HG49NT4M/qyFVVNkmyj6CVqQInGJBvSVI5iHDsN0kLRjRNoC3aDuDjim4ivvtzlMgFZfH/FtQp9LfkolcSgDeY7CI5X508m4qKjO/1YtRI1AGFT2l1fm9+ffvHyzSkdVSz1mOB+0VebRjb+mwC+ANAucNx/CRVF9I3UfwFQSwMEFAAAAAgAAABCUJ9CxOhWAAAAVwAAACAAAABpbmZlcmNoYW9zLTAuMS4wLmRpc3QtaW5mby9XSEVFTA3JMQ6AIAwAwJ1X8IESFSc+YNyMMTqjaYCEtKbAwO/11rsiYoYTpSQmp0czqAUJxVcWp6OvT8yJwh92MrPamSusBbYmmNPtdJWG6vDB6bdbICYET119UEsDBBQAAAAIAAAAQlDxO2nQLAAAADMAAAArAAAAaW5mZXJjaGFvcy0wLjEuMC5kaXN0LWluZm8vZW50cnlfcG9pbnRzLnR4dItOzs8rzs9JjS9OLsosKCmO5crMS0stSs5IzC9WsFVAcPSSczKtchMz87gAUEsDBBQAAAAIAAAAQlBx/FYgdgIAACwEAAArAAAAaW5mZXJjaGFvcy0wLjEuMC5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRV1SS2/bMAy+61cQPbWA0Q097LCbEiuNNr8gK81yVGw51uBYgaUs6L8f6aTtOiCAIZLfi0wuNWSusWOwjC396XVyhz7CffMAT1+fvkHhj2aEH6YzE2OVnY4uBOdHcAF6O9n9KxwmM0bbJtBN1oLvoOnNdLAJRA9mfIWTnQIC/D4aN7rxAAYa1GE4GXukCb6LFzNZHG7BhOAbZ5APWt+cj3aMJpJe5wYb4D72Fu7qG+LuYRZprRmYG4F6by24uNj7c4TJhji5hjgScGMznFvy8NYe3NHdFAg+hw8MSc8BE5DPBI6+dR197RzrdN4PLvQJtI6o9+eIxUDFeYsJ5fjiJwh2GBgyOPQ9Z/1wN8+Q9RMtNN5WFKhy6f3xcxIXWHeeRpS0M6b1uLJZ8bdtIlVovPPD4C8UrfFj6yhR+M6YxpbZ+z92znK97egjWr1aoAOcPq56a4XeDAPs7W1hqIvrNf/EmUg+RDy8MwOc/DTr/R/zEfXXAupypbdcCZA1VKp8kalI4Y7X+L5LYCv1utxowAnFC72DcgW82MFPWaQJiF+VEnUNpWIyrzIpsCaLZbZJZfEMC8QVJf6BZS41kuoSSPBGJUVNZLlQyzU++UJmUu8StpK6IM5VqYBDxZWWy03GFVQbVZW1QPkUaQtZrBSqiFwU+hFVsQbiBR9Qr3mWkRTjG3SvyB8sy2qn5PNaw7rMUoHFhUBnfJGJqxSGWmZc5gmkPOfPYkaVyKIYjV3dwXYtqER6HH9LLcuCYizLQit8JphS6XfoVtYiAa5kTQtZqTJPGK0TEeVMgrhCXFlo1fDpIjhC700t3gkhFTxDrprAFPFt+JH9BVBLAwQUAAAACAAAAEJQnKl7YOwBAAAeAwAAIQAAAGluZmVyY2hhb3MtMC4xLjAuZGlzdC1pbmZvL1JFQ09SRH3SuZaqQBAG4HyeBZQGoSG4AZuILCoDgiQchWYRBKYbFH36a+IZJzGpoILvP7VUbY5wWh47Mk+Sqq2GJJn1d4qUR5YX/lnb8bytrrdscbQS27EFkEtdmaLrwDlt4zMm29ixqKJUCijAS1/Vr5Y21RuUWWN5ViaS8UZuCdDq2IcDwb0WNypcliucnbsJKvSkmTrFshL3LqGpR7i6oHZ4A7WfNFBayco0x4EWTcs31/AnHDsNW9iRJ3oMc9iss9UwFhRgJPhHxKjv8LsmjKM1JUOweJaj4R62Tp3s/M5cra3avRLJSvdBuGRlblNQnCD8GZQ03ZtUc1fRlUzW3y2jY4TQYdrub/Bxi5e41IfL2LnfkrHgQhktntICvEk0MwMzZpZVZKCf3W7u6L6syb78sjGKXbBjGXxCIdAiuQJFmKePaVTvJThw0c+SWzOYrIqWoSADF5/scKXr9gsOubzu6wukb7l0VUw+piFhUFgjhyb9CEVTdNFJEjWkF5QIP7HPI+F70ndVO5DZMA2/a/GncXWyuChQWSPRfQnTm5aUxPYss4+NZhd4oNaKKicU/3EpTZWiliAyt01Vd7/1V0KEgOAPLFYeYStf8v22e5yLlB+nk2IUscHLoinThyArlvrzIQTxU4anqxtPo6iv/1BLAQIUAxQAAAAIAAAAQlDLGZMxhAAAAJ8AAAAWAAAAAAAAAAAAAACkgQAAAABpbmZlcmNoYW9zL19faW5pdF9fLnB5UEsBAhQDFAAAAAgAAABCUEMUVGGtAwAA9QgAABEAAAAAAAAAAAAAAKSBuAAAAGluZmVyY2hhb3MvY2xpLnB5UEsBAhQDFAAAAAgAAABCUBw2J3VnDgAA3SoAABgAAAAAAAAAAAAAAKSBlAQAAGluZmVyY2hhb3MvZXhwZXJpbWVudC5weVBLAQIUAxQAAAAIAAAAQlB1MXc26AUAAFUOAAAUAAAAAAAAAAAAAACkgTETAABpbmZlcmNoYW9zL3JlcG9ydC5weVBLAQIUAxQAAAAIAAAAQlBUjgAf4QQAADkOAAARAAAAAAAAAAAAAACkgUsZAABpbmZlcmNoYW9zL3Nsby5weVBLAQIUAxQAAAAIAAAAQlDOFLDOMgwAAKIbAAAjAAAAAAAAAAAAAACkAVseAABpbmZlcmNoYW9zLTAuMS4wLmRpc3QtaW5mby9NRVRBREFUQVBLAQIUAxQAAAAIAAAAQlCfQsToVgAAAFcAAAAgAAAAAAAAAAAAAACkAc4qAABpbmZlcmNoYW9zLTAuMS4wLmRpc3QtaW5mby9XSEVFTFBLAQIUAxQAAAAIAAAAQlDxO2nQLAAAADMAAAArAAAAAAAAAAAAAACkAWIrAABpbmZlcmNoYW9zLTAuMS4wLmRpc3QtaW5mby9lbnRyeV9wb2ludHMudHh0UEsBAhQDFAAAAAgAAABCUHH8ViB2AgAALAQAACsAAAAAAAAAAAAAAKQB1ysAAGluZmVyY2hhb3MtMC4xLjAuZGlzdC1pbmZvL2xpY2Vuc2VzL0xJQ0VOU0VQSwECFAMUAAAACAAAAEJQnKl7YOwBAAAeAwAAIQAAAAAAAAAAAAAApAGWLgAAaW5mZXJjaGFvcy0wLjEuMC5kaXN0LWluZm8vUkVDT1JEUEsFBgAAAAAKAAoA6gIAAMEwAAAAAA==", "llmgateway-0.1.0-py3-none-any.whl": "UEsDBBQAAAAIAAAAQlDhr0mJfgAAAJIAAAAWAAAAbGxtZ2F0ZXdheS9fX2luaXRfXy5weR2MwQrCQAwF7/2KR85a6gd46EUoCArtfQndtQ3UXUlii39v9TgDM0TUdpjY08YfaHm75Ak8ajGDpeVxnIt5iuAc0d47vLSsEpMaNvEZw3AZ0F9vdoDuEyzyFP9BcpVk/yymSTmyS8kYZ5ZsNRFVVQjr/tltCDiDmvpUN1R9AVBLAwQUAAAACAAAAEJQz5ofi4wMAAC8JgAAEQAAAGxsbWdhdGV3YXkvYXBwLnB5nRpdc6PI8V2/YkIeDHsIy/Y6dcdFqdv1+ZJLXdkuW3lIOS4KwWCxRqCDwbai6L+nu2cGGEBr7/EgiaGnu6e/u5FlWYsVZ4+h4C/hltkX11cXlzeLO28ds4pHIi3yip1O/+L4bMPLqeB5mAtWAjjL0nUqXLYpi+c05iWLVmGas5dUrCYlF2XKK5claVmJqSieeM5EuuZFjTsAUbsrLaM6FWxZ8vAJ7oucLRa/LNjmh3N2zHhZFiWRcyeVAJB1mj+yTVhVYlUW9ePKZcdrJBZVnmVZk0lSFmsWBEkt6pIHAUvXm6IULMzzQoR0mslEr1XbPEoLfVuGeVys9R0yK5FFRZZpQaiHF0WdC17K53EowigDjnjzvFlCAfAsbkiuhNi8ym1JWIlwk+otv8Dtp5tfXXbLf695JQwgr+TVBui3FP55d311qxZddpOB5Bf8VbRLd1pYekmJJsvWSteelrg+lFTEZ7k6gCZtN/QXZAe/4RrATm4vF7f//vT5t0s2Z7uPs+9d9nH2A3ycnuMH/DqfzfDjFD/O8OPjfhLcLT4tLoOL659pmxVlRcVjy2cAahUbnsPPE/i5CrMkUPen+8lk8lMj3wl9shtlTBdFnqSP/oTBlYdr7jOwGbpbhhUP6jJrV9ZFzDu3IOXgiW9pgf2PXRU5B6bwa4zg36VQuvS0QVc+i9NI3AMet8fYAwGSm/jgPRUByUXpVwEaus+SrAgFUD+febPu02UNztQ+Ppn1nhfP4C5AzmBB1JuM39MeV259eIDNZJd2zJOwzkSQhJEoyu0ctzlSPOFrEArB1xtRBeCwgT6fz9IcqZ8quUZPRZIEJN+q5W3myedRkefgPIHy/WBdGfxL9ilIBBQkRgHPNaCMAEEaZ3wU8GzWQGZFIERiPv5ePVSGH7yk4PEvgYHAAAD/CapwDQKsmmP3IEBMFKN6qpt55wZcVBQZ0Mq7xE5MEBDwEmRYRxHHWKIJnoH9gZpYsAorQANxJxd2tKrzJ58tt4JXDpv+jS0Bv7RDiIKLsuYsTSDosbu7S0bALApLDMksZHmRT1GvW6bQMfAEEVL8RAwQvOsyZ8sjSz23fOsIuJGIIBZtsjDi9tJilsuWluUAodgEB3iIt1/dA2F9ad3/fH11+WA1gHBW6WDBJ2l7zZmuwR/rjTQApgzTZxgVKiYggeknwMlLiIEqAQL4gIyrOalkhzIFIibBBmmeiiCwK54lLouyFAB9Gaq9T5gjLmgJ8pbfc2g4ShFvpbfBzuTRNyODI9kngwTkXqQQ0c1GfSMK9RMwgMY11EaiJ8QmHkwHmkMd33tBy4CXDkZnr1TkIdPBQHD/YMKS1zRW2ouEFClRJCQ5lD5JjUywDZztqUHQt2RNFRqjTOYAB2nJRQMg3JDwldFT7UBKI+5A30wUI0pc1knCSx57k4bQQgOZtQZsgZhYMQ6fW7HC2kHldApLPfTkJ2IFx1bO0qDXlO2QlUXGp0WebTu+FRfALRp8hIWB8yMZH7pZW7KUMq1XEn9KRtugf1kBUlY2yftllUYrBIqKsgQ+feJS2gVoSRKj0yw5kAJnF4yD7HjZODFeKx5iQqLkGtZiVZTpf6kGgkSaWJ95WEL230lj9FQC3Ft7DB3mIuMZGNhu32BGu0S0Hz50TNiilAq41Wa6bfeIGexAvcADYL/I08h22qfltjWbThQK0Zt1qebhTQAnJqvzqCiwJXF1Vsh3M8fV6p9rr/LGUwwUmJiEWi74a8Q3UN1IiEu0V5MrLRrUFSoIlW6avL4k4w2wF1J50zlx55DWiOlafW8LGnczw446ub4VM+W7B51yRNS/g2o6Ecpb1mkWB8pmbevm+m4BcTuxtLHocmp//HxyDAWNOI4KTJNUJQPkl6rI54Zi5urbPH8ry3lXYIqLiuexDUy4KrrPMbENlCXD4D8WixvSF0iM8VFTSizl9j7bie2G29zxggDrxCDYt17TVbEH5beoMe/GnP0Js//MRI2Focm7VDawG/d0/S0GkVh4HuWbfUb2wD/SvfdPTmcPXsxx0abgWs2PVKo9cjpHkt6CpbrWs+QAV6Bweekw0UasObMEtBTHEDtzoSIZZeoWgdKp98jBSFSMnKJowQQgxx+2OAhwEPBQnUPHkSHVkCmxDroKc2AIMrUz2DRIcxC6Nmg+dDeEByWj67anBcMZVljOkDu8qGLrm+GdKDZULfwKvFKUNTdjz/rtpvttRiOP83Xz7iT5kXjMphBAJj3EY7m/xMBAub9lGLNe1FiIUXQY/G6x92DRHzOPAbC+TLs5kDH6hjSSKsabjEGq+Eb168u0nZ5EqARupJnmYZZt32kPskmIALvgAdi+PVKNYkJQcwZfNb0YeNWSLVKR8bnV9vzKg1XdMR+WxLYWnnyi0qatwuwcpTlsAJUkXYYxco62BQVPCYqZn8HiBhoZWnQk8UyOGIC6MXLA43mdppmKZK/bJxsrTWfsdFsuqozknMAcfhD2TgvZsIzr/e7RXO20jO6oAQzttrvbaCdNzG0Laa73+sahhVIxCqekLgs2NpOKvWrRaZgFBYQaa4FU1C8VZ+BISZY+rsRhkJIShhyhebf0ZYOzNE1WXrzYZH9Umvj9CDMoC3vRBpw+oLmJ3VY+hE5ON3S3qEodqhdc7J/ddszyMNqV2EpuSZ3prrIz0USpuXodxC5zrGMU2f2n2Py73a4KxZ+6jQZ4Xq8xQnAyMjpUL81A3xctsQ/s6uoe9z+4jenKe7OYEjiVhX1Lqha4TUI37QGLGwk21zO2YQJWBnFvW9VTCmk0ltM3eQbngX03Zycju3KR5jU/TK6d41GbTv3S0oMYB15ERqwYfhdH0uqXdbX9I3yhTp5QG2Cxj1ITBwdeI/xouHGKjbtIHY0DYdWozdYe6/hHMk1ZtunNo25gtLBBOGhNxtsSEskSkge0ljHFNyl1MHNdFYBsZt5IppPSbDShO9o35I+X7uXcAx71pgino5ptjkFYxsy9x7Kk9gbDEkjXj9DzIDDW3PBgb41K/Il9x07YX9lX7YiMvueb2FMccsOBuOmVyjukbRZAVcY5VQOeOatlH5h9yj58YE8O/px553AICOOeDOLA3IgFZtUBkxqWNSCWVIklA0s9FOx6p0wgHCCfh87ZqUixxRgxJQL9CbQnu5Jj6FEysVJ6azOKXLadQSraWbLVsnzQzBM2MsQ23DZHgDUdhmF5l/usp1XKuODLGGE0pAdly7qynf1+wKJ6gWW5zQgooCnofPBep38KtbN7jCzNeWXO9ZAd+wnKFSVTlz1TlV6UgkOHpKSvGewpKAuXPMNmEAdnFlk7YKKA3ipLDogsbeqWiQE5avzpqK0vgx2i2geiEGG22+2I1H6uPM7a79nueX/kGOeQASQaFa3/TrK6ciKd7Xaaa4Nw5+XUfc9pH7o84YWvKintYgD14G4sFB1mh8Iu7AoqiGZ5XB3gCKmA9OmrHTwpyV+FV1afrcMUm/B6gFQv/PZ1IItqrLJQC6pCJxF9sypUfU5FAGR1ZQlyFViSPyRTUMdZCs4aqOBNCiX/Am3IO0hoQJOGib/TI8myNOCvq7CuGvxs18a0ox7EkcuOjsCIOqlEjwH/k1velwKqXCLnQPjEpU682BQ4jhsduvVDAwLYKkf7+q1yRy/qTf5cT6bNYc6retOP8S/Mi3y7Luqqw7Gan2jdk15sbRdjZVyJBY8Gx1y2DcIE+wdz09hco/uu297JJI7Ruf0TAmjLxWkVceyro+1d1hmazek1tB5G7iidbqfEAk3Dd2Xoe2cJ2EHLg5pyy3yqxYSjze7cOpDTU2K/3xQcqncapGZfY4hX7Rmv5DoZs2ddcvI2Uh28IUzQYPsKmyVhmklcmnWAaU+hCylfnacnanzP/2bv26riddq607RDD7o2W986HbX8mf2r4uU0CSMc3NGfRcoaejl6s6PUxPAFzjM4IhZdaR5lNTT/TNVQtCiP2Bzqxw56egOl/hxR458ucAH/tKJ1QjTtprCTtYSD6RvLUY4UK3jSqtcbfSdjHLzJnb4ymvfJZVTQ5lbML9N1Jc3cVp24NFws/XCy4XsnYPrvwKXo9nCq1W4DYWLet2/q2gj1iA2Madejwz25g4aLNOKDnKMJ0hTywJR2bKaG1+hcDa/RpqPrkuijMjyqaXg/xkkXG/z1xqaTulCtxWkY4Gx23jnAG2P04Qze+cpLFfU+YmmpRHIfdcWHoiNeHga5x4gL5ptl+5RGX5KvOVJwVBQcYUTPFWXFpIdUEDPaNR28umvtmK/d0cKpp5MOv7B18n9QSwMEFAAAAAgAAABCUBx90GB4BAAAUw8AABUAAABsbG1nYXRld2F5L2JyZWFrZXIucHmtV0tz2zYQvutXrNkL2TCs1MSHKJEP9TSnTtJpNJODRsOhSdDCGCJUALRz6I/vAiBBgA/bM63GDwn8sI8Pu99CURT9xRmjzT3s95/38AsQIbiAJ9pU/AmKpoICSirKliq4E6R4IAIu+lfwR1rhm/j265fb3//cf8vOFUhSKsobCZsU3idZFEWrVS34GfK8blUrSJ4DPV+4UGi64aow6A5TYiD9/g5Ukb9bslqtSlZICV2k301s2xXgqyI12qYNVXkeS8LqVMfAm0puoWa8UInF6Zd+nHVPYdfjwse5LM4XRnC3cX1QLX46GEupNXg84l7zME5WLoaiqjr3qnOcwmPBWtKHAW9v4AtvyHbeX1ZcLqSp4rjfl3jG84toGzKyP2Px6UQZCe2aEwxWDusj/sAnUPA2oGSwM43uwvFfrfyMm7l4aKPGCdrYVeKWBcFCaIARa8E58WwbBuScA0alssdxfK2jwyPUWNE5EovxhYkdB5+XD9dzDs07+GfENAZoSghrlNiDz7qYVTIEQJElrgw4JLeLTNt06w9oL9akaDi6hk0CP8M6+3DtIIyncKKIQ5rjhySFM21i+x7e6JYLt4+J0E8OjB8Ra1CHEz0isF/W7uIHXGA8cS13a1v/N9v5Ngvs6pJxSSrND8eyxcLDP0ig1ZAb+PbHV0DKrZSIQhFcOxc/IL6nj4jEsN2hfzQmjRW0dipYnZsPRa1QXErOGfZ689F7grDOvcWgEt2RXLZlSaTEir/nvLKLMrXGhQuUo2GoC2yTDmIlaklJGM+VqlUuXU9bXXQLeBbv1qmf0FYfDi7/qpeLH7nhINccDFvW2XUa1IN59bn6xjfX6Ti/3sG7ibLp8jDvvHD6lSASXbkutSD6ccyhhyHCzuyY+h34kNHT0JZ2jvhA0uOe3STE8laV/GzsL+AR+RNssjUiTPApcqw/dM7HmqssCV0ZRyNvWCikyoteBrrmR3zQr73o6ByRwJrR+5PO6DO20zyK6xZfj2gQRSOpmXlbK27BxJFKmHlz8IQql0RNZo1JaavxM3MBdcimfLXzGZgV/I4a83/63AvXn1gG7Qlf4HSHRGtOo+2k5kO+0a2ah8wT+LqTGHgjWqIxopcHqY7do+NqKJU5HXdLDezCgs0afx7pcYBzd9yiz5rUorobOibTU8ozicVuqvwch36HYYQ3usaPINYWqTSDyZS1viDotRunIkmn3f3SSMbmqgYrEsvAHrM3xwXeLERlhGZSsYGwLl2PTM76buU2LEhDD8K2T5aOUReiGyPRXB4vdXNn1EYCn3aOsoXSHur2DWr5BOPic7Cb3aywTs0PLgRB9nOrhDJOnkH2p9SVcgglmOdSFqPjXd5i4a7P1LQUTB293ICzZ7v5T2f7fyQxIlsjn6vcybwaarq7c7920IV8uM0uMEPCHK04EBZlzQnz8CVh0OKxRLhvCwOkL1bv4vIM78PpTC6mXlDeNyrGcLSbRphL7A6dTjOzPCDiKqgHLWizHT57LV+e4L4y7EVLxomYtX8BUEsDBBQAAAAIAAAAQlAW8ScGsAMAAC4IAAARAAAAbGxtZ2F0ZXdheS9jbGkucHmNVd2L4zYQf/dfoerJPrKur2WhDbhwFNqXvbuF7ssRgk6xx46wLLmSnNzS3v9+ow87ydI96hCsGc33bzxDKf0s5dhzB2f+TO7uGq060ZP+XD7zUX7eEjMr4o5AFpHO6JFw8und+wfSCQklpTTLApexbnazAcaIGCdtHOFKaced0Mpm2cIz/cSNhYXWNmpP3B2lOCyqj0iuOj6W5OQSbcmnaZH+M7J+D9FvyKPRJ9GCWejGAAowVMiyrIWOSM1bFnPNveMtsc6Qf4PXgtz9dmtwmxF8Wt2QOoRSWt4B8zZyrxAsFCX6aJmDLy4viqAwpSgsqv3zNbA6bYjiI2wIhq68yR1dxei+FA5GmxfRoX8GeEZtbUtQJ2G0Kntw+bSjfBIM7xiy6b4goiNTuLq5KAhICy9uaLEaXz3vfEx7dHRbuNyz6xjvgVtgs5E1Ol/OGLDBuokppz/SYkNG3UIQCAe635Dks8Z/dOtAceV8RTD3GFdi0Q0WKQohQnyWDoXSXRRMbC9IDeJDt+S+2hB6mDEKJN5WVTKgT2AM5hEq77Yk7xAsl592UW+Psa6cqL0vigCO25CTR2ZxnADxBXbkh5qsQUQ8D9eJHLADBjBXiQxnH8CwDUAP++Bh8OZzOvIvjDu0PjnLJjBsAYP6jHgz6K5jodC+MhR7VUHjmBMj6NmxMXA7gbEzpwdQNzcrwv/9UAQN+MhEK+GFRSs1c64LZMh6SH0a8zWAH7i6/TrytY3q9YSf3JELVUthXR66PNCh8rG0zCNRRxRSTS/ovB5/Ug6gvdRekFxdrG1Qr6fvmE7wsbNQrT4zWx8irAuN5fm5QuOL3CgUs3ycJKyiVyyU/qn6XiarGWwEDE6bWJHF0g0XjVXl/ZXvRmuJQalLlBcOCr+9/z+eEa0DttfcNGDtJYkXbJ82en7zZjgXaXqOCGWOc/y0JR7gHbbTHofnB60A+92/wgz1hzjJ0pSeT6LRRmVxOqLosgvKd6afR1Du0VMGIbUNjhW/OGr68PB+3T5n4Y7EVwQdj8Jhn2FHGoG4ko5L6b+b2HkWd09Lnp7+eLr76+EjTpTe8DZsojT/ppK3LePJb06X1Ue9yb9nYaCtn8wMr0kftfWTKDVfTasy/F617guA8u55glood9H8pfq1ikocKzKVoSBe1YYSx6tUuRK3cX5ZZvn1GuNlPBQIlg+u5qV/4aZBz0j41wYXX88knHBM0zM3SqieelTxQ2fMD3rc3TVOOcY8xozRiF8AvMi+AVBLAwQUAAAACAAAAEJQylhRud8CAAAtCAAAFAAAAGxsbWdhdGV3YXkvbGltaXRzLnB5jVXLbtswELzrKxbKxWpk1e3RqIOiQXsq0gD1zTAE2lo5rClSoKgmaZp/75LUW2kQHWyR3MfMcHcVhuEt6qVByaQBo84ol4f6eEYDmhkEwQtuuDzB4vrHzfXX2+3PpMigwqPhSsLHKIEtLxB4BSWrKsyAywAfSsGP3IhHqBSYO4qiTvxojQxWhh0Ewj03d6o2UAnEkuInYRgGQa5VAWma16bWmKbAi1JpA0xKZZjNWDU2GTPsKGzGqjXqtmLIOYosCILP3V7gfmFr+X1x9NYB0GM5riEXihkYPBdeCSKF2pJVMnPmR1YyIvY4cbmAQ60rIsP/oLPz3q3VxgNacMnN5hsTFUbOiiCZ1uYv3CiJnWmGOauF2djNGIaOzpOOSaZSVSa1Z2m6qFDkkedkH7tMGg4bn8KZJC2DYSCNORfCnccg1X2DKYLllUPVh+W5j2yR2+uka/G4mcysJ1z1573XHFDB5RhOPDK4hIUNtuyDRfDOL+yFRWOaDszGpZ/ic9iQdIOCPSy6I8dyoIBhZ5zRp/f+Aj8kKyfHQSkxUbmVz4WcCtUQuiJ4/9djSaejQ41U/xK2usZgsueqoEdOu/oxZblB/TYCbv0WBhTd9rAVtr0ID3eKaJWsLN3W/tMGVl7ydud9f3PUk00junnz3Q4X1OtBTQ/KOYamC9JBl/abk16MQf1GrXmG1HcZP5pdZXQMpi4F7hoL97ff981m/6Zd08Sn08Uw/zzxpAy79OTZvysNT89jQz9eRygHc2lP7lMPmplj+/6NS0vIufQq+gyNiH60r4HMXQXMZmBTr80nwLY0l2Ok4+J0YpAIlHVMPDlRUh8mHmkZzUu/Cb3z5pbBANeiy/GCpxNj4PcUMiHUPWbhGlYxhBp/0dfJL5+nxfpS7l44F2gu22wojqeAOrdKtMJ73yhxc2XUVXMGuw6+vQSK5XqnZ7GHS+reKQ91fm0GvIZ9MgCGukzhDwM7Fv8AUEsDBBQAAAAIAAAAQlBd9FH52gcAAG0QAAAjAAAAbGxtZ2F0ZXdheS0wLjEuMC5kaXN0LWluZm8vTUVUQURBVEHFV9ty3LgRfedXdNmVtVQZzkW2fBlFrpW1ckq7kqzYs7vZUrlmMCQ4hEUSDADOaHal13xAPjFfktMA52JJiR9jlyQSaDa6T5++4Fw6kQon4l+ksUpXQ9rr7kcXopRDKopyJpxciGW03u13B91+9KkpS2GWQzo6pVaEjG6cqmYkEqOtJSuLLM61dTIlUaV0dHlKtdFzlUIVLZTLaTR6P6JPZx9shwyUUKFK5fhFOqOk9Z+lcmbYQBxOSS5UZaOjxuXaxLIUqhjShS5FRT+KTBj6S9Xg5Qs/21zkv4v0+xlLdRNdvo3OVCIrK+OTm9pIG9w5Px2t19+rAk6fnR6fXHw6iT7KfzQKcvHlEsdB9O3h8+6gv1n/QVk3pExYJ2r19hDAPNzNnatveG/v1f2terkUZfH28OX9jWauEm0q/9Wb6DJAZmG1M2IIPOYPNTlp3dvD1wckWYgOD+kZ5J7dFzRNlrHal98SdPpaVlNZJTl9TzPl/sx+2GGvh+e8mTKcPY+7h723Jf5A8w/SJkbVHL/4WFdOVi4eLWsA7SDZA4uuU72ooujpNt2i/wOvuhGD/UUmjgbPSWc02CdV0ZWqMmngnIzxZERca+MyXSj9eed/w/JfP9zFUU+f0q+5cORyZWnaqCK18JpsKYqCPtSyOjqNoa+GhdNCrsFw/E0pruFFphvDnpfK4s1p7EkSdV2oJLgF4zMDyNkXxTyRJgb+AitbsNCODx9sSK6l69CLvTcBxclHwLWMjzInzWR3g57flDe1rhBLJQqaiuRaZ5mH9YtyECeczuJLwbZnyMAG5OpAgrRBnBC6DH7yhwH8VXDXgex4bWzxaokSZZJGOZoaCf8R69SoORu+JAGSFAWTxIe+frNPYsYxdXwkmEA7rA6S0hhtvPvwyHsiKFPGujig4FQpwTeyGhu20AvanC+qSjvKdZHygcgYZB0xE8VMdunq+MPF8cnl6FO3TD/vbL3sMlgFW0NSIEVKCZcrZUvv4oIDCq8S6LErXix5pRQMpY1WJQCoNnwU+GtAiEYYRFLKtEs/W+CUiYQB8A4GEe2zThRB688W30bRZDKZCptHtaqJAWK6Pfl2im9y80m0eaYYHK0yNaPZosvlDAvMcnrdf9Pno/x5vIH0bkUiH/AhXRU6EUWHQip/pkf/PSXmXMpU9sQ52EK+1HNgwcXjUbpFay4NIyJ/2JDoDzgvx40pQm2Gv4O9V90+/g+Gr/v9fgdqU4ndvy1k1eNfaIfxoLv/Lj4FWqZJ3B3UBaOH99UxfmgGXXkjyrqQDOJaodWljP0zuF2r8bVcjmU1H9LJ34/OL89Oxj+d/HYXhfz0JqcyE02BvP2D6Yq+DOOmjeH6/KJ/9zVKLSaWMwYVEkFJW2Gy6nd5QG3eH9LNqgTkUgAcnDMDoYesZn3O9kH9uwhpMHYuc+PSDullv/94qNamtJnok3CKGIUSp1HSrC9Rq7g8s6tUjnwGjn0GjtsM9IfBlAengUhIe2pqjjqym7OkrF2ojDbXC0uVRjr5VsMpjzOrqD0JYC5UBc6Mof05R1tVY+uDhYUBFhINOMCpsT/+LpD4ZC4NGpG0KHpWohIYXwknN/EmF9aFatK5t9FaaCc+4b/eY1zj0k4OaNIrucAmEOPiyjX9q/K3CnGnTfF1Qe6sayke7bWqUaJzjDIe8s4KY0Kqo+bv9CkpoDylHg28BB72KBdoqvzGdf5eKe0g++KsULMc3WGrh+BQvZBpz0jumTINJTuUc3mTi8Zy+bFbvQ40wF6O+h9Ft3Su0wZW3tJlY9hfuo1u4zj2P9iebFDqhk41gexkxCR553sVAz3ytpzxPpBHLwN7qBaWPeSyqxLliuUBU8IWUtbsGGzggcnu0v1zWqz8QR8DDL96ukAz8xlsNp7DBaY+umgzzQK0yXHoTu9WGnZamP/9z38FmHVoROg+mx7U8dtr8Elwr11TsMOpMgUPWCioe2gyGj6bO4JR72EUT0Pt1rCNxUIU16HVbQjzSMfz+ZOwh5azp0JMIREH0TadwC8HB0uf3wDZ5RjOZjnSKFD3gXVJodi6347Oz6jtFOzM5K9h/9gvcRw330wgiMsFqHTrmTPiUEXR4FUIWte7uuo/YJS8kSZRPt74BmMQ/1l1fxgoedBBwbjaTKnfGtw2kruo3phRNvPlDspm4XIwyg8HHlYUSSqtz5cDygpxvQzr+/0/4advd4db85aHjlnkiRuSGGUANSpej0RiarWZ+m7uC0/b0Ta5zGThbXBqM/B47ky2ClqoN5jGs0wl3C7xyeOxz7Txw8PaAtCmqbjqiHU3SQqJarvf36PVxLyV5W3u6yppjOG6G3hiueosWbOVtWAQulsTiL9XUVwSprg5df3v775Do2xMIsN7Dyj0ROLUnAHE5vbYEkt60r3CLePzE7/lr0GhYIM3v4hCtfP9ehYE2phXV9GMog+VxKWoWg2C87Ozc4j4CZpE6FYers0Q6OdI5oRlgXXh3WUmwkM/a/NnAEj7cdbfJ3YGL3jczBXGPx/DoHt1FEys2xroR0LhkpxHeu6bLdM3zZJ9SFyoqheYR1WVFE0q0yg6xnTpQ8yXAnREIxfGE67jp8uYZ5V0dZ1CxBpmnWtvCyGvQlz9qMUesZG6KpZU49qSLHn68prEgufLsLiWm86VbjgfQW2+XAcT28t1FPFN+z9QSwMEFAAAAAgAAABCUJ9CxOhWAAAAVwAAACAAAABsbG1nYXRld2F5LTAuMS4wLmRpc3QtaW5mby9XSEVFTA3JMQ6AIAwAwJ1X8IESFSc+YNyMMTqjaYCEtKbAwO/11rsiYoYTpSQmp0czqAUJxVcWp6OvT8yJwh92MrPamSusBbYmmNPtdJWG6vDB6bdbICYET119UEsDBBQAAAAIAAAAQlCFGi7/LAAAADMAAAArAAAAbGxtZ2F0ZXdheS0wLjEuMC5kaXN0LWluZm8vZW50cnlfcG9pbnRzLnR4dItOzs8rzs9JjS9OLsosKCmO5crJyU1PLEktT6xUsFVAcPSSczKtchMz87gAUEsDBBQAAAAIAAAAQlBx/FYgdgIAACwEAAArAAAAbGxtZ2F0ZXdheS0wLjEuMC5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRV1SS2/bMAy+61cQPbWA0Q097LCbEiuNNr8gK81yVGw51uBYgaUs6L8f6aTtOiCAIZLfi0wuNWSusWOwjC396XVyhz7CffMAT1+fvkHhj2aEH6YzE2OVnY4uBOdHcAF6O9n9KxwmM0bbJtBN1oLvoOnNdLAJRA9mfIWTnQIC/D4aN7rxAAYa1GE4GXukCb6LFzNZHG7BhOAbZ5APWt+cj3aMJpJe5wYb4D72Fu7qG+LuYRZprRmYG4F6by24uNj7c4TJhji5hjgScGMznFvy8NYe3NHdFAg+hw8MSc8BE5DPBI6+dR197RzrdN4PLvQJtI6o9+eIxUDFeYsJ5fjiJwh2GBgyOPQ9Z/1wN8+Q9RMtNN5WFKhy6f3xcxIXWHeeRpS0M6b1uLJZ8bdtIlVovPPD4C8UrfFj6yhR+M6YxpbZ+z92znK97egjWr1aoAOcPq56a4XeDAPs7W1hqIvrNf/EmUg+RDy8MwOc/DTr/R/zEfXXAupypbdcCZA1VKp8kalI4Y7X+L5LYCv1utxowAnFC72DcgW82MFPWaQJiF+VEnUNpWIyrzIpsCaLZbZJZfEMC8QVJf6BZS41kuoSSPBGJUVNZLlQyzU++UJmUu8StpK6IM5VqYBDxZWWy03GFVQbVZW1QPkUaQtZrBSqiFwU+hFVsQbiBR9Qr3mWkRTjG3SvyB8sy2qn5PNaw7rMUoHFhUBnfJGJqxSGWmZc5gmkPOfPYkaVyKIYjV3dwXYtqER6HH9LLcuCYizLQit8JphS6XfoVtYiAa5kTQtZqTJPGK0TEeVMgrhCXFlo1fDpIjhC700t3gkhFTxDrprAFPFt+JH9BVBLAwQUAAAACAAAAEJQvyPLd+sBAAAaAwAAIQAAAGxsbWdhdGV3YXktMC4xLjAuZGlzdC1pbmZvL1JFQ09SRH3Sy5KiMBiG4X1fC1EIB2ExCwhBYRTtFgXcUIABIwdpCApe/fQsrGE23sDz/vXVX5ZVHjPyiMd5FNGasiiaNSPXXWIoK7/048iD3Chp417i9XmTbVGfwPq3pLijlJfPUWd3tOGtHcCcICkf5T8tbpoJBKJydIIzLE4VDFdVCL/jrdOYSd72wYUlhTJ0FQX9YImY0zThPylpSVyQdqJF/QP6ZEBqlERIsoZAXF6pg299oOzcltdO+ap2DDzuyIETNShOtbSkE8mMddOlG5iyHe8OONZ2SbBGdvAQjgyMHjOeJSoQSazU5iCvSVOppBVl3QSrQ2UhZF1vyZ6IRCl3TtZqrMvVc791YLm8wS87IW2E8gD/xaZnAX4mzPjZmXYM0Dq7zTfY003d01/2Cu6tM++B817aywYkn1cvGdqjm2L9+r2/WrgS/cNdN1wx5STIy+9sf4Xx+gX7YlY0RbUAj0y7G7Z8AouOJ35BNqBr+oVqqy5JNNUkOOfUxTuW1Kwdo+ZG659V2MBeheazPnzG1PEViHw/REflVip1n93NXgbCQdTjIQX+ehlaIScL7wolTUndkW6+thF29/hVCIigeAy2xtOv9So77m7Pa57K/ZAYy/y0lHXV1kF4OOfWz5vyivqu8YXR9svkuI8/UEsBAhQDFAAAAAgAAABCUOGvSYl+AAAAkgAAABYAAAAAAAAAAAAAAKSBAAAAAGxsbWdhdGV3YXkvX19pbml0X18ucHlQSwECFAMUAAAACAAAAEJQz5ofi4wMAAC8JgAAEQAAAAAAAAAAAAAApIGyAAAAbGxtZ2F0ZXdheS9hcHAucHlQSwECFAMUAAAACAAAAEJQHH3QYHgEAABTDwAAFQAAAAAAAAAAAAAApIFtDQAAbGxtZ2F0ZXdheS9icmVha2VyLnB5UEsBAhQDFAAAAAgAAABCUBbxJwawAwAALggAABEAAAAAAAAAAAAAAKSBGBIAAGxsbWdhdGV3YXkvY2xpLnB5UEsBAhQDFAAAAAgAAABCUMpYUbnfAgAALQgAABQAAAAAAAAAAAAAAKSB9xUAAGxsbWdhdGV3YXkvbGltaXRzLnB5UEsBAhQDFAAAAAgAAABCUF30UfnaBwAAbRAAACMAAAAAAAAAAAAAAKQBCBkAAGxsbWdhdGV3YXktMC4xLjAuZGlzdC1pbmZvL01FVEFEQVRBUEsBAhQDFAAAAAgAAABCUJ9CxOhWAAAAVwAAACAAAAAAAAAAAAAAAKQBIyEAAGxsbWdhdGV3YXktMC4xLjAuZGlzdC1pbmZvL1dIRUVMUEsBAhQDFAAAAAgAAABCUIUaLv8sAAAAMwAAACsAAAAAAAAAAAAAAKQBtyEAAGxsbWdhdGV3YXktMC4xLjAuZGlzdC1pbmZvL2VudHJ5X3BvaW50cy50eHRQSwECFAMUAAAACAAAAEJQcfxWIHYCAAAsBAAAKwAAAAAAAAAAAAAApAEsIgAAbGxtZ2F0ZXdheS0wLjEuMC5kaXN0LWluZm8vbGljZW5zZXMvTElDRU5TRVBLAQIUAxQAAAAIAAAAQlC/I8t36wEAABoDAAAhAAAAAAAAAAAAAACkAeskAABsbG1nYXRld2F5LTAuMS4wLmRpc3QtaW5mby9SRUNPUkRQSwUGAAAAAAoACgDnAgAAFScAAAAA"}
for _n, _b in _W.items():
    open('/kaggle/working/wheels/' + _n, 'wb').write(base64.b64decode(_b))
print('wheels:', sorted(os.listdir('/kaggle/working/wheels')))

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap,driver_version --format=csv

In [ ]:
# vLLM on the Kaggle image: the cu129 release wheel with the matching PyTorch index (the default wheel targets
# CUDA 13), and the image's preinstalled torchaudio/torchcodec removed because they are built for other CUDA
# versions and vLLM imports them (torchcodec is re-pulled by vLLM's requirements, hence the second uninstall).
%pip uninstall -q -y torchaudio torchcodec
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.31.0/vllm-0.31.0+cu129-cp38-abi3-manylinux_2_28_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip uninstall -q -y torchcodec
%pip install -q "git+https://github.com/NomanJafar/servellm" "tokenbench[plots] @ git+https://github.com/NomanJafar/tokenbench"
# Private repos: wheels are embedded in the auto-generated cell above and written to /kaggle/working/wheels.
!pip install -q /kaggle/working/wheels/*.whl
import os; os.makedirs("/kaggle/working/results", exist_ok=True); os.chdir("/kaggle/working")
import torch, vllm, inferchaos.slo, llmgateway.app, servellm.launch, tokenbench
assert torch.cuda.is_available()
import vllm.multimodal.media.audio
print("vllm", vllm.__version__, "torch", torch.__version__)

## A real replica as provider p0, a mock as p1

In [ ]:
import json, subprocess, time, httpx, threading
from servellm.launch import ServerConfig, VllmServer
from inferchaos.experiment import Experiment, Providers, _free_port, _wait_health
from inferchaos.slo import SLO

MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
class RealAndMock(Providers):
    """p0 = vLLM on the GPU (killable, restartable with its real cold start); p1 = tokenbench mock fallback."""
    def __init__(self):
        super().__init__(n=2, capacity=16, ttft_ms=40, itl_ms=2)
        self.vllm = None; self.cold_starts = []
    def start(self):
        self._spawn("p1")
        self._start_vllm()
    def _start_vllm(self):
        t0 = time.monotonic()
        self.vllm = VllmServer(ServerConfig(model=MODEL, dtype="float16", max_model_len=4096, gpu_memory_utilization=0.9, port=8000),
                               log_path=f"results/startup_{len(self.cold_starts)}.log")
        self.vllm.start(ready_timeout_s=1200)
        self.cold_starts.append(time.monotonic() - t0)
        self.urls["p0"] = "http://127.0.0.1:8000"
    def kill(self, name):
        assert name == "p0"
        self.vllm.proc.kill(); self.vllm.proc.wait(timeout=10)
    def restart(self, name):
        assert name == "p0"
        self._start_vllm()
    def stop(self):
        super().stop()
        if self.vllm: self.vllm.stop()

prov = RealAndMock(); prov.start()
print("vLLM cold start:", round(prov.cold_starts[0]), "s")

In [ ]:
slo = SLO(target=0.99, ttft_s=0.8)
exp = Experiment(prov, slo, rate=4.0, slo_ttft_ms=800, breaker_min_samples=10, breaker_window_s=15, breaker_cooldown_s=10, max_tokens=32)
# model name differs per provider: the gateway rewrites it from each ProviderConfig
exp.cfg.providers["p0"].model = MODEL
res = exp.run(fault="kill", baseline_s=60, fault_s=60, recovery_s=240)
res["vllm_cold_starts_s"] = prov.cold_starts
prov.stop()
json.dump(res, open("results/kill_real_vllm.json", "w"), indent=1, default=str)
print({k: res[k] for k in ("requests", "errors", "fallbacks", "budget_consumed", "peak_burn_rate", "breaker_opened_at", "provider_restarted_at", "recovery_s")})
print("cold starts (s):", [round(c) for c in prov.cold_starts], "| alerts:", res["alerts"])

In [ ]:
from inferchaos.report import write_results
out = write_results({"kill (real vLLM on T4)": res}, "results")
print(out["table"])
from IPython.display import Image, display; display(Image("results/faults.png"))

In [ ]:
!cd results && zip -qr ../results.zip . && ls -la ../results.zip